In [15]:
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, LSTM
from tensorflow.keras.optimizers import RMSprop

In [16]:
filepath = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

In [17]:
text = open(filepath, 'rb').read().decode(encoding='utf-8').lower()

In [18]:
text = text[0:800000]

In [32]:
characters = sorted(set(text))

In [20]:
char_to_index = dict((c,i) for i, c in enumerate(characters))
index_to_char = dict((i,c) for i, c in enumerate(characters))

In [21]:
SEQ_LENGTH = 40
STEP_SIZE = 3

In [22]:
sentences = []
next_characters = []

In [23]:
for i in range(0, len(text) - SEQ_LENGTH, STEP_SIZE):
  sentences.append(text[i:i+SEQ_LENGTH])
  next_characters.append(text[i+SEQ_LENGTH])

In [24]:
x = np.zeros((len(sentences), SEQ_LENGTH, len(characters)), dtype=np.bool)
y = np.zeros((len(sentences), len(characters)), dtype=np.bool)

In [25]:
for i, sentence in enumerate(sentences):
  for t, character in enumerate(sentence):
    x[i, t, char_to_index[character]] = 1
  y[i, char_to_index[next_characters[i]]] = 1

In [26]:
model = Sequential()
model.add(LSTM(128, input_shape=(SEQ_LENGTH, len(characters))))
model.add(Dense(len(characters)))
model.add(Activation('softmax'))

model.compile(loss='categorical_crossentropy', optimizer=RMSprop(learning_rate=0.01))

model.fit(x, y, batch_size=256, epochs=4)

Epoch 1/4
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 2.0195
Epoch 2/4
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 1.6302
Epoch 3/4
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.5312
Epoch 4/4
1042/1042 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - loss: 1.4785


In [27]:
model.save('textgenerator.h5')

In [28]:
model = tf.keras.models.load_model('/content/textgenerator.h5')

In [29]:
def sample(preds, temperature=1.0):
  preds = np.asarray(preds).astype('float64')
  preds = np.log(preds) / temperature
  exp_preds = np.exp(preds)
  preds = exp_preds / np.sum(exp_preds)
  probas = np.random.multinomial(1, preds, 1)
  return np.argmax(probas)

In [38]:
def generate_text(length, temperature):
  start_index = random.randint(0, len(text) - SEQ_LENGTH - 1)
  generated = ''
  sentence = text[start_index: start_index + SEQ_LENGTH]
  generated += sentence
  for i in range(length):
    x = np.zeros((1, SEQ_LENGTH, len(characters)))
    for t, character in enumerate(sentence):
      x[0, t, char_to_index[character]] = 1

    prediction = model.predict(x, verbose=0)[0]
    next_index = sample(prediction, temperature)
    next_character = index_to_char[next_index]

    generated += next_character
    sentence = sentence[1:] + next_character
  return generated

In [39]:
print('-------------0.2--------------')
print(generate_text(300, 0.2))
print('-------------0.4--------------')
print(generate_text(300, 0.4))
print('-------------0.6--------------')
print(generate_text(300, 0.6))
print('-------------0.8--------------')
print(generate_text(300, 0.8))
print('-------------1.0--------------')
print(generate_text(300, 1.0))

-------------0.2--------------
ose his head ere give consent
his master to the presse of the hands of thee,
and the hand the suit and the state of thee,
and the brother and the grace to her hand to thee,
and the hand to heart to the triend of thee,
and the hand the sent the presse and heart,
to have have a seet and the suit of the heart,
and the mistress and the marria
-------------0.4--------------
ream the boar did raze his helm;
but i do not the house of this amberit.

camillo:
you may deserved the brother his souls, and thee seed me to as
the husband the days of the greate to her a cast.

leontes:
the brother were in the minder to the honour.

gloucester:
i have doth the hand thee, to the grace.

king richard ii:
we have stay the
-------------0.6--------------
thers beat aside the point.

gloucester:
shall call the company stort to dead to well
a man out a bounted.

gloucester:
a stand me the seet other stand, we not?
how the part me to thee, stay, me she fair breath me.
now, when 